# ultrasonic รอบสี่ — เทรนบน Colab (GPU)

**ก่อนเริ่ม**
1. เมนู **Runtime → Change runtime type → T4 GPU** แล้วกด Save
2. อัปโหลด `ultrasonic_r4_colab.zip` ไปไว้ใน Google Drive ที่โฟลเดอร์ **`MyDrive/ultrasonic_r4/`**
3. รันทีละเซลล์จากบนลงล่าง (Shift+Enter)

ผลทุกการทดลองเก็บลง Drive (`MyDrive/ultrasonic_r4/runs/`) ถ้าเซสชันหลุด รันเซลล์ใหม่ได้เลย
การทดลองที่เสร็จแล้วจะถูกข้าม · **ปล่อยแท็บนี้เปิดไว้** Colab ฟรีตัดเซสชันถ้าไม่มีความเคลื่อนไหวนาน

## 1. ตรวจว่าได้ GPU

In [ ]:
import torch
print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ไม่มี GPU — ไปที่ Runtime > Change runtime type > T4 GPU แล้วรันเซลล์นี้ใหม่")

## 2. เชื่อม Google Drive แล้วแตกไฟล์

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, zipfile
DRIVE_DIR = '/content/drive/MyDrive/ultrasonic_r4'
ZIP = f'{DRIVE_DIR}/ultrasonic_r4_colab.zip'
RUNS = f'{DRIVE_DIR}/runs'
WORK = '/content/r4'
assert os.path.exists(ZIP), f'ไม่พบ {ZIP} — อัปโหลด zip ไปที่โฟลเดอร์นี้ใน Drive ก่อน'
os.makedirs(RUNS, exist_ok=True)
if os.path.exists(WORK):
    shutil.rmtree(WORK)
zipfile.ZipFile(ZIP).extractall(WORK)
# ผลเขียนตรงลง Drive ผ่านลิงก์ จะได้ไม่หายตอนเซสชันหลุด
os.symlink(RUNS, f'{WORK}/car/data/r4_runs')
os.chdir(f'{WORK}/car')
os.environ['MAP2_ROUND'] = 'r4'
os.environ['PYTHONIOENCODING'] = 'utf-8'
print('พร้อม:', sorted(f for f in os.listdir('.') if f.endswith('.py')))

## 3. ทดสอบสั้น ๆ (~1-3 นาที)

เทรน 1 รอบด้วยข้อมูลช่วงเดียว ดูว่าทุกอย่างทำงานและ **หนึ่งรอบใช้กี่วินาที**
(บนเครื่องที่บ้าน รอบเต็มใช้ ~110 วินาที · ส่งตัวเลขนี้ให้ Claude ประเมินเวลาได้)

In [ ]:
!python train_r4.py --name colab_smoke --train 1 --epochs 1 --threads 2
for ext in ('json', 'pt', 'log'):
    p = f'{RUNS}/colab_smoke.{ext}'
    if os.path.exists(p):
        os.remove(p)

## 4. รันคิวการทดลอง (ทั้งหมด ~1.5-2 ชั่วโมงบน T4)

ชื่อขึ้นต้นด้วย `gpu_` เพื่อไม่ชนกับผลที่รันบนเครื่อง · ค่าตั้งต้น = สูตรรอบ 3
เปลี่ยน **ทีละอย่าง** จากค่าตั้งต้น (ดูเหตุผลใน `car/ROUND4.md`)

- `gpu_base_s0/1/2` ค่าตั้งต้น 3 ค่าสุ่ม = ความต่างที่เกิดจากความบังเอิญ · s0 เทียบกับบนเครื่องด้วย
- `gpu_loss_*` วิธีวัดความผิด 5 แบบ · `gpu_act_*` ฟังก์ชันกระตุ้น 6 แบบ
- `gpu_norm_log` บีบคลื่นแบบลอการิทึม · `gpu_slice5` ไอเดียตัดช่วงเวลา 5 ms เลื่อนทีละ 1 ms
- `gpu_lc_*` เส้นโค้งการเรียนรู้ (เทรน 1, 2, 3 ช่วง) · `gpu_long40_base` เทรน 40 รอบ

In [ ]:
import subprocess, sys, time
QUEUE = [
    ("gpu_base_s0", []),
    ("gpu_base_s1", ["--seed", "1"]),
    ("gpu_base_s2", ["--seed", "2"]),
    ("gpu_loss_bdh", ["--loss", "bdh"]),
    ("gpu_loss_mse", ["--loss", "mse"]),
    ("gpu_loss_bce_l1", ["--loss", "bce_l1"]),
    ("gpu_loss_dice_huber", ["--loss", "dice_huber"]),
    ("gpu_loss_focal", ["--loss", "focal"]),
    ("gpu_act_leaky", ["--act", "leaky"]),
    ("gpu_act_elu", ["--act", "elu"]),
    ("gpu_act_gelu", ["--act", "gelu"]),
    ("gpu_act_silu", ["--act", "silu"]),
    ("gpu_act_mish", ["--act", "mish"]),
    ("gpu_act_tanh", ["--act", "tanh"]),
    ("gpu_norm_log", ["--norm", "log"]),
    ("gpu_slice5", ["--slice-ms", "5", "--slice-step-ms", "1"]),
    ("gpu_lc_s1", ["--train", "1"]),
    ("gpu_lc_s12", ["--train", "1,2"]),
    ("gpu_lc_s123", ["--train", "1,2,3"]),
    ("gpu_long40_base", ["--epochs", "40"]),
]
EPOCHS = "12"

for name, extra in QUEUE:
    if os.path.exists(f'{RUNS}/{name}.json'):
        print('ข้าม (เสร็จแล้ว):', name)
        continue
    cmd = [sys.executable, 'train_r4.py', '--name', name, '--epochs', EPOCHS, '--threads', '2'] + extra
    print('=' * 70, '\nเริ่ม', name, ' '.join(extra), flush=True)
    t = time.time()
    with open(f'{RUNS}/{name}.log', 'w', encoding='utf-8') as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
        p.wait()
    print(f"{'เสร็จ' if p.returncode == 0 else 'ล้ม'} {name} · {(time.time() - t) / 60:.0f} นาที")

## 5. สรุปผลทั้งหมด

In [ ]:
!python r4_report.py

## 6. ส่งผลกลับ

ไฟล์ผลอยู่ใน Google Drive ที่ `MyDrive/ultrasonic_r4/runs/` (`*.json` คือผล · `*.pt` คือโมเดล)
ดาวน์โหลด `*.json` ทั้งหมดไปวางที่ `car/data/r4_runs/` ในเครื่อง แล้วบอก Claude